# Loan Eligibility Prediction — Cleaned Notebook

**Author:** Adib Raihan Ashidiq (Boekanadip)

**Purpose:** Notebook yang terstruktur untuk analisis data, pembersihan, pembuatan fitur, dan pemodelan (RandomForest).

**Ringkasan singkat:** Saya membersihkan notebook original untuk keperluan portofolio: memperbaiki alur, menambahkan reproducibility (random_state), ringkasan dataset, pipeline sklearn, evaluasi model, dan instruksi menjalankan.

## 1. Setup dan Reproduksibilitas
- Pastikan file dataset berada di `data/loan_data_2007_2014.csv` relatif terhadap root repo.
- Notebook ini menggunakan RandomForestClassifier (random_state=42).


In [ ]:
# Imports dan versi paket
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print(f'python: {sys.version}')
print('pandas:', pd.__version__)
print('numpy:', np.__version__)


## 2. Data loading (relatif) dan quick overview
- Jika dataset tidak tersedia di folder `data/`, jalankan `scripts/download_data.py` atau tempatkan file secara manual.


In [ ]:
from pathlib import Path
data_path = Path('data/loan_data_2007_2014.csv')
if not data_path.exists():
    print(f'File {data_path} tidak ditemukan. Pastikan dataset diletakkan di folder data/.')
    df = None
else:
    df = pd.read_csv(data_path, on_bad_lines='skip')
    print('Loaded', df.shape, ' — showing a concise preview')
    display(df[['loan_amnt','term','int_rate','grade','sub_grade','emp_length','home_ownership','annual_inc','purpose','loan_status']].head())


## 3. Cleaning & feature selection (ringkas)
- Hapus kolom identitas/indeks yang tidak dipakai.
- Hapus kolom yang 100% missing.
- Berikan strategi impute sederhana untuk fitur numerik dan kategorikal.


In [ ]:
if df is not None:
    # Drop index-like columns dan kolom all-NaN
    df = df.copy()
    for col in ['Unnamed: 0']:
        if col in df.columns:
            df.drop(columns=col, inplace=True)
    # Drop columns that are completely empty
    all_null_cols = [c for c in df.columns if df[c].isnull().all()]
    if all_null_cols:
        print('Dropping all-null columns:', all_null_cols)
        df.drop(columns=all_null_cols, inplace=True)

    # Select a manageable set of features for demonstration
    features = ['loan_amnt','term','int_rate','grade','sub_grade','emp_length','home_ownership','annual_inc','dti','purpose']
    target = 'loan_status'

    # Keep only rows where target is in common categories (e.g., Fully Paid / Charged Off / Current)
    df = df.loc[df[target].notna(), features + [target]].copy()
    print('After selection:', df.shape)
    display(df.head())


Notes:
- Untuk portofolio, saya memilih subset fitur yang mudah dijelaskan. Anda dapat menambah fitur lebih lanjut di bagian feature engineering.
- Untuk demonstrasi training cepat, kita akan sample data jika dataset terlalu besar.


In [ ]:
if df is not None:
    # Simplify target to binary: Fully Paid vs Charged Off/Default (example).
    df = df[df[target].isin(['Fully Paid','Charged Off','Default','Does not meet the credit policy. Status:Charged Off','Does not meet the credit policy. Status:Fully Paid'])].copy()
    df[target] = df[target].apply(lambda x: 'fully_paid' if 'Fully Paid' in x else 'charged_off')

    # Sample (10%) for faster demonstration if dataset large
    if len(df) > 50000:
        df = df.sample(frac=0.1, random_state=RANDOM_STATE).reset_index(drop=True)
        print('Sampled down for demo, new shape:', df.shape)

    # Train/test split stratified
    X = df.drop(columns=[target])
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    print('Train/Test sizes:', X_train.shape, X_test.shape)


## 4. Pipeline & Modeling (RandomForest)
- Preprocessing: impute numeric (median) + scale; impute categorical (most frequent) + one-hot.


In [ ]:
if df is not None:
    numeric_features = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
    # exclude target-even if present
    # keep only small number of categorical features for demo
    categorical_features = [c for c in X_train.select_dtypes(include=['object']).columns.tolist()]

    num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())])
    cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('ohe', OneHotEncoder(handle_unknown='ignore'))])

    preprocessor = ColumnTransformer([('num', num_transformer, numeric_features),('cat', cat_transformer, categorical_features)])

    clf = Pipeline([('preproc', preprocessor),('clf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))])

    print('Fitting model (this may take a moment)')
    clf.fit(X_train, y_train)
    joblib.dump(clf, 'model_randomforest.joblib')
    print('Model saved to model_randomforest.joblib')


## 5. Evaluation
- Tampilkan metrik utama dan confusion matrix.


In [ ]:
if df is not None:
    y_pred = clf.predict(X_test)
    y_proba = None
    try:
        y_proba = clf.predict_proba(X_test)[:,1]
    except Exception:
        pass

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label='fully_paid')
    rec = recall_score(y_test, y_pred, pos_label='fully_paid')
    f1 = f1_score(y_test, y_pred, pos_label='fully_paid')
    roc = roc_auc_score(y_test.map({'charged_off':0,'fully_paid':1}), y_proba) if y_proba is not None else None

    print(f'Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}, ROC-AUC: {roc}')

    cm = confusion_matrix(y_test, y_pred, labels=['charged_off','fully_paid'])
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=['charged_off','fully_paid'], yticklabels=['charged_off','fully_paid'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion matrix')
    plt.show()


## 6. Kesimpulan & Next Steps
- Ringkasan hasil di atas.
- Next steps: feature engineering lebih dalam, cross-validation, hyperparameter tuning, SHAP untuk interpretability, deployment sebagai API.
